In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import joblib
import datetime
import warnings
warnings.filterwarnings('ignore')

# Load features and target from parquet files
features_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/features/features.parquet"
target_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/features/target.parquet"

X = pd.read_parquet(features_path)
y_loaded = pd.read_parquet(target_path)

# Convert target back to Series (since it was saved as DataFrame)
y = y_loaded.iloc[:, 0] if isinstance(y_loaded, pd.DataFrame) else y_loaded

print("Data loaded successfully from parquet files!")
print(f"Feature set shape: {X.shape}")
print(f"Target variable distribution:\n{y.value_counts()}")
print(f"Target proportion:\n{y.value_counts(normalize=True)}")
print(f"\nFeature columns: {list(X.columns)}")

Data loaded successfully from parquet files!
Feature set shape: (39856, 14)
Target variable distribution:
active_after_week
0    21940
1    17916
Name: count, dtype: int64
Target proportion:
active_after_week
0    0.550482
1    0.449518
Name: proportion, dtype: float64

Feature columns: ['total_posts_week1', 'avg_posts_week1', 'max_posts_week1', 'std_posts_week1', 'active_days_week1', 'slope_week1', 'cv_week1', 'first_active_day', 'last_active_day_week1', 'is_high_frequency', 'is_consistent', 'day0_engagement', 'first_3days_ratio', 'zero_post_days']


In [2]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Important for imbalanced datasets
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Testing target distribution:\n{y_test.value_counts(normalize=True)}")

Training set shape: (31884, 14)
Testing set shape: (7972, 14)
Training target distribution:
active_after_week
0    0.550496
1    0.449504
Name: proportion, dtype: float64
Testing target distribution:
active_after_week
0    0.550426
1    0.449574
Name: proportion, dtype: float64


In [3]:
# Scale the features for models that require it
scaler = StandardScaler()

# Fit on training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for better visualization
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print("Feature scaling completed!")
print("Scaled training set statistics:")
print(pd.DataFrame(X_train_scaled).describe().loc[['mean', 'std']])

Feature scaling completed!
Scaled training set statistics:
                0             1             2             3             4   \
mean -2.228524e-19  3.164505e-17 -1.002836e-18  1.103120e-17 -1.314829e-16   
std   1.000016e+00  1.000016e+00  1.000016e+00  1.000016e+00  1.000016e+00   

                5             6             7             8             9   \
mean  4.100485e-17  1.105348e-16  6.952996e-17  2.718800e-17  1.114262e-17   
std   1.000016e+00  1.000016e+00  1.000016e+00  1.000016e+00  1.000016e+00   

                10            11            12            13  
mean  3.565639e-17 -3.610210e-17  1.949959e-17  1.314829e-16  
std   1.000016e+00  1.000016e+00  1.000016e+00  1.000016e+00  


In [ ]:
# Initialize multiple models to compare performance
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    # 'SVM': SVC(probability=True, random_state=42)
}

# Handle class imbalance by computing class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print(f"Class weights: {class_weights}")

# Apply class weight to Logistic Regression
models['Logistic Regression'].set_params(class_weight='balanced')

Class weights: [0.90827256 1.11233603]


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [5]:
# Train and evaluate each model
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}")
    print(f"{'='*50}")
    
    # Use scaled data for LR and SVM, original for tree-based models
    if name in ['Logistic Regression', 'SVM']:
        X_tr = X_train_scaled
        X_te = X_test_scaled
    else:
        X_tr = X_train
        X_te = X_test
    
    # Train the model
    model.fit(X_tr, y_train)
    
    # Make predictions
    y_pred = model.predict(X_te)
    y_pred_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'auc': auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    if auc:
        print(f"AUC: {auc:.4f}")
    
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy')
    print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")


Training Logistic Regression
Accuracy: 0.6859
AUC: 0.7462
Cross-validation accuracy: 0.6846 (+/- 0.0067)

Training Random Forest
Accuracy: 0.6722
AUC: 0.7246
Cross-validation accuracy: 0.6784 (+/- 0.0102)

Training Gradient Boosting
Accuracy: 0.6914
AUC: 0.7524
Cross-validation accuracy: 0.6944 (+/- 0.0092)
